In [ ]:
import torch
import json

In [ ]:
reference_path = r'/root/autodl-tmp/Blip2CC/.cache/coco/annotations/coco_karpathy_test-raw.json'
with open(reference_path, 'r') as j:
    gt_raw = json.load(j)
test_path = r'/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/20250206233/test_epochbest_rank0.json'
with open(test_path, 'r') as j:
    test_data = json.load(j)

In [ ]:
# 计算分类指标
nochange_caps = ["there is no difference", "the two scenes seem identical", "the scene is the same as before", "no change has occurred", "almost nothing has changed"]
pre = []
truth = []

for gt, test in list(zip(gt_raw, test_data)):
    label = gt['changeflag']
    test_cap = test['caption']
    # test_cap = test
    if test_cap in nochange_caps:
        pre.append(0)
        truth.append(label)
    else:
        pre.append(1)
        truth.append(label)
# with open('cls-data.json','w') as f:
#     json.dump(pre,f)

In [ ]:
data = []
for gt, test in list(zip(gt_raw, test_data)):
    item = {}
    label = gt['changeflag']
    test_cap = test['caption']
    # test_cap = test
    if test_cap in nochange_caps:
        pass
    else:
        item['image_A'] = gt['image_A']
        item['image_B'] = gt['image_B']
        item['conversations'] = [{"from": "human", "value": "<image> <image> Please judge whether these two images have changed. Please answer yes or no."}, {"from": "gpt", "value": "Yes"}, {"from": "human", "value": "If changes have occurred, count the number of road and building changes separately."},{"from": "gpt", "value": "No roads has changed, and no buildings has changed."}]
        data.append(item)

In [ ]:
"""
{"id": 9, "image": ["train/A/train_000001.png", "train/B/train_000001.png"], "conversations": [{"from": "human", "value": "<image> <image> Please judge whether these two images have changed. Please answer yes or no."}, {"from": "gpt", "value": "No"}, {"from": "human", "value": "If changes have occurred, count the number of road and building changes separately."}, {"from": "gpt", "value": "No roads has changed, and no buildings has changed."}, {"from": "human", "value": "Based on the above analysis, please describe the changes of these two images in detail."}, {"from": "gpt", "value": "No change has occurred."}]}
"""

In [ ]:
def process(data):
    # data = [{"id": 9, "image": ["train/A/train_000001.png", "train/B/train_000001.png"], "conversations": [{"from": "human", "value": "<image> <image> Please judge whether these two images have changed. Please answer yes or no."}, {"from": "gpt", "value": "No"}, {"from": "human", "value": "If changes have occurred, count the number of road and building changes separately."}, {"from": "gpt", "value": "No roads has changed, and no buildings has changed."}, {"from": "human", "value": "Based on the above analysis, please describe the changes of these two images in detail."}, {"from": "gpt", "value": "No change has occurred."}]}]
    annotation=[]
    import copy 
    for d in data:
        # print(d)
        dialog = copy.deepcopy(d)
        all_turns = dialog['conversations']
        all_turns = [
                    {
                        "answer": all_turns[d+1]['value'],
                        "question": all_turns[d]['value'],
                    }
                    for d in range(0, len(all_turns),2)
                ]
        
        for i in range(len(all_turns)):
            dialog_instance = copy.deepcopy(dialog)
            dialogue_context = ' '.join([f" q: {t['question']} a: {t['answer']}" for t in all_turns[:i]]).strip()
            last_turn = all_turns[i]

            question = last_turn["question"]
            answer = last_turn["answer"]
            if dialogue_context=='':
                dialog_instance["question"]= 'q: '+ question
            else:
                dialog_instance["question"] = dialogue_context + '  q: ' + question
            dialog_instance["answer"]=answer
            dialog_instance['conversations'] = ''

        annotation.append(dialog_instance) ##重点
    return annotation

In [ ]:
make_test = process(data)
with open('open_dig1.json', 'w') as f:
    json.dump(make_test,f) 

In [ ]:
##读出第二阶段的结果
with open('/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/20250319020/test_epochbest.json','r') as f:
    ccap = json.load(f)
data2 = []
for d,r in zip(ccap, make_test):
    item={}
    item['image_A'] = r['image_A']
    item['image_B'] = r['image_B']
    item['conversations'] = [{"from": "human", "value": "<image> <image> Please judge whether these two images have changed. Please answer yes or no."}, {"from": "gpt", "value": "Yes"}, {"from": "human", "value": "If changes have occurred, count the number of road and building changes separately."},{"from": "gpt", "value": d['caption']},{"from": "human", "value": "Based on the above analysis, please describe the changes of these two images in detail."}, {"from": "gpt", "value": "No change has occurred."}]
    data2.append(item)

make_test2 = process(data2)
with open('open_dig2.json', 'w') as f:
    json.dump(make_test2,f) 

In [ ]:
# 最后的文件
with open('cls-data.json','r') as f:
    su = json.load(f)
with open('/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/20250319023/test_epochbest.json','r') as f:
    cap = json.load(f)

i = 0
re = []
for it in su:
    if it==0:
        re.append('there is no difference')
    else:
        re.append(cap[i]['caption'])
        i+=1

In [ ]:
with open('all_result.json', 'w') as f:
    json.dump(re,f) 

In [ ]:
with open('all_result.json', 'r') as f:
    data3 = json.load(f)
with open('all_result.jsonl', 'w') as f:
    for j in data3:
        f.write(j+'\n')

In [ ]:
re = r'/root/autodl-tmp/Blip2CC/.cache/coco/annotations/coco_karpathy_test-raw.json'
with open(re, 'r') as j:
    s = json.load(j)
print(s[113])
print(test_data[113])

In [ ]:
#loc
re = r'/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/loc/test_epochbest.json'
with open(re, 'r') as j:
    s = json.load(j)
print(s[113])

with open(r'/root/autodl-tmp/GPT-api/LEVIR-MCI-dataset/LevirCCcaptions-loc.json', 'r') as f:
    data_t = json.load(f)
datajson_t = [x['change_loc'] for x in data_t if x['filepath'] == 'test']
print(datajson_t[113])

In [ ]:
#count
re = r'/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/count/test_epochbest.json'
with open(re, 'r') as j:
    s = json.load(j)
print(s[113])

with open(r'/root/autodl-tmp/GPT-api/LEVIR-MCI-dataset/LevirCCcaptions-loc.json', 'r') as f:
    data_t = json.load(f)
datajson_t = [x['change_counts'] for x in data_t if x['filepath'] == 'test']
print(datajson_t[113])